In [38]:
import pandas as pd
import numpy as np
import os

In [14]:
## Loading  data

In [3]:
df = pd.read_csv("C:/Users/Bhanu Bisht/Data_Analytics_Project/data/raw/02_nav_history.csv")

In [4]:
print(df.head())

   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


In [5]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB
None


In [6]:
print("Shape:")
print(df.shape)

Shape:
(46000, 3)


In [7]:
print("\nData Types:")
print(df.dtypes)


Data Types:
amfi_code      int64
date          object
nav          float64
dtype: object


In [9]:
df.tail()

,amfi_code,date,nav
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873
45999,149324,2026-05-29,279.7511


In [10]:
df.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [11]:
df.columns

Index(['amfi_code', 'date', 'nav'], dtype='object')

In [12]:
df.dtypes

amfi_code      int64
date          object
nav          float64
dtype: object

In [16]:
# 1. Parse dates
df['date'] = pd.to_datetime(df['date'])

In [17]:
df['date']

0       2022-01-03
1       2022-01-04
2       2022-01-05
3       2022-01-06
4       2022-01-07
           ...    
45995   2026-05-25
45996   2026-05-26
45997   2026-05-27
45998   2026-05-28
45999   2026-05-29
Name: date, Length: 46000, dtype: datetime64[ns]

In [18]:
# 2. Sort by amfi_code and date
df = df.sort_values(['amfi_code', 'date'])

In [19]:
df.head()

,amfi_code,date,nav
5750,100016,2022-01-03,520.4608
5751,100016,2022-01-04,515.0971
5752,100016,2022-01-05,521.7239
5753,100016,2022-01-06,515.7880
5754,100016,2022-01-07,515.1639


In [20]:
# 3. Remove duplicates
df = df.drop_duplicates(subset=['amfi_code', 'date'], keep='last')


In [24]:
print(df.columns.tolist())

['amfi_code', 'date', 'nav']


In [27]:
# 4. Create complete daily date range for each fund
def fill_missing_dates(group):
    idx = pd.date_range(
        start=group['date'].min(),
        end=group['date'].max(),
        freq='D'
    )

    group = (
        group.set_index('date')
             .reindex(idx)
             .rename_axis('date')
             .reset_index()
    )

    # Fill amfi_code
    group['amfi_code'] = group['amfi_code'].ffill().bfill()

    # Forward-fill NAV for weekends/holidays
    group['nav'] = group['nav'].ffill()

    return group

df = (
    df.groupby('amfi_code', group_keys=False)
       .apply(fill_missing_dates)
       .reset_index(drop=True)
)

C:\Users\Bhanu Bisht\AppData\Local\Temp\ipykernel_11664\347068120.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(fill_missing_dates)


In [28]:
# 5. Validate NAV > 0
invalid_nav = df[df['nav'] <= 0]

print("Invalid NAV rows:", len(invalid_nav))

# Keep only valid NAVs
df = df[df['nav'] > 0]

# Final check
print(df.info())
print(df.head())

Invalid NAV rows: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64320 entries, 0 to 64319
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       64320 non-null  datetime64[ns]
 1   amfi_code  64320 non-null  float64       
 2   nav        64320 non-null  float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 1.5 MB
None
        date  amfi_code       nav
0 2022-01-03   100016.0  520.4608
1 2022-01-04   100016.0  515.0971
2 2022-01-05   100016.0  521.7239
3 2022-01-06   100016.0  515.7880
4 2022-01-07   100016.0  515.1639


In [29]:
print("Missing dates:", df['date'].isna().sum())
print("Missing NAVs:", df['nav'].isna().sum())
print("Duplicate rows:",
      df.duplicated(['amfi_code', 'date']).sum())
print("NAV <= 0:",
      (df['nav'] <= 0).sum())

Missing dates: 0
Missing NAVs: 0
Duplicate rows: 0
NAV <= 0: 0


In [30]:
print(pd.__version__)

2.2.3


In [33]:
#Count Missing Values

In [31]:
df.isnull().sum()

date         0
amfi_code    0
nav          0
dtype: int64

In [34]:
#Percentage Missing

In [32]:
(df.isnull().sum()/len(df))*100

date         0.0
amfi_code    0.0
nav          0.0
dtype: float64

In [35]:
#Numerical Summary

In [36]:
df.describe()

,date,amfi_code,nav
count,64320,64320.000000,64320.000000
mean,2024-03-16 12:00:00,120247.000000,269.530464
min,2022-01-03 00:00:00,100016.000000,26.136600
25%,2023-02-08 18:00:00,118632.750000,69.180175
50%,2024-03-16 12:00:00,119551.500000,122.751200
75%,2025-04-22 06:00:00,120842.250000,260.171425
max,2026-05-29 00:00:00,149324.000000,4268.549700
std,NaN,14352.272787,577.144541


In [37]:
df.dtypes

date         datetime64[ns]
amfi_code           float64
nav                 float64
dtype: object